> **TrustBreast — Notebook 1.** Tables 2–6, 8, 10, 12; Figs 3–5.

# File 1 — FIXED version (Objective 1 + reviewer fixes)

## Before running
1. In Colab, select **Runtime → Change runtime type → GPU**.
2. The folder `MyDrive/TrustBreast_locked/` must exist in Google Drive (the saved 99.12% model). This notebook **does not retrain** the model — it only LOADS it, so the 99.12% result does not change.
3. Click **Runtime → Run all**. Allow the Drive mount permission if prompted.

## Expected runtime
The full notebook takes about **3–4 hours** (longest steps: STEP 7 leakage ~1 hour, STEP 9 repeated splits ~1–2 hours). Keep the Colab tab open.

## At the end
The last cell is **STEP 11 — RESULTS SUMMARY**. Copy and share **only that cell's output** (plus a screenshot of any cell that raises an error).

In [ ]:
# Colab: clone the repo (it contains the models/ folder). On local Jupyter this cell does nothing.
import os
if os.path.exists('/content') and not os.path.isdir('models') and not os.path.isdir('../models'):
    !git clone -q https://github.com/Iqra672-ai/TrustBreast.git /content/TrustBreast
    %cd /content/TrustBreast
    !pip -q install -r requirements.txt


## STEP 1 — Determinism (run first)

In [ ]:
# ============================================
# CELL 0 — DETERMINISM  (run this FIRST, before any imports)
# Adds three settings that make the DNN identical across runs:
#   1. os.environ flags   -> make GPU/cuDNN deterministic (must be set BEFORE imports)
#   2. all seeds          -> python / numpy / tensorflow
#   3. enable_op_determinism() -> LOCKS GPU floating-point order (this was the missing piece)
# The reseed() helper is used later to reset the RNG right before the DNN is built.
# ============================================
import os
os.environ['PYTHONHASHSEED']         = '42'
os.environ['TF_DETERMINISTIC_OPS']   = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import random, numpy as np, tensorflow as tf
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("enable_op_determinism() ON  ->  DNN now reproducible")
except Exception as e:
    print("Note: enable_op_determinism unavailable (older TF). The remaining fixes still apply.")

def reseed(s=SEED):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)

print("TF:", tf.__version__, "| Determinism setup done. Now run the remaining cells.")


## STEP 2 — Libraries install

In [ ]:
!pip -q install scikit-learn xgboost imbalanced-learn tensorflow scipy shap lime dice-ml anthropic matplotlib seaborn statsmodels

## STEP 3 — Data load (all 569 patients — for CV and the remaining tests)

In [ ]:
import pandas as pd, numpy as np
from sklearn.preprocessing import LabelEncoder
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
col_names = ['id','diagnosis',
  'radius_mean','texture_mean','perimeter_mean','area_mean','smoothness_mean','compactness_mean',
  'concavity_mean','concave_points_mean','symmetry_mean','fractal_dimension_mean',
  'radius_se','texture_se','perimeter_se','area_se','smoothness_se','compactness_se',
  'concavity_se','concave_points_se','symmetry_se','fractal_dimension_se',
  'radius_worst','texture_worst','perimeter_worst','area_worst','smoothness_worst',
  'compactness_worst','concavity_worst','concave_points_worst','symmetry_worst','fractal_dimension_worst']
df = pd.read_csv(url, header=None, names=col_names)
df['diagnosis'] = LabelEncoder().fit_transform(df['diagnosis'])   # B=0, M=1
X = df.drop(['id','diagnosis'], axis=1); y = df['diagnosis']
feature_names = list(X.columns)
print(f"Data: {len(df)} patients | B: {(y==0).sum()} | M: {(y==1).sum()}")


## STEP 4 — LOAD the locked 99.12% model (no training, no saving)

In [ ]:
# ===== LOAD the locked 99.12% model (run this FIRST) =====
# Requires the saved model folder in Google Drive: MyDrive/TrustBreast_locked/
# (produced once by File 1 - Objective 1). No retraining here, so the number is always 99.12%.
import os, pickle, joblib, numpy as np, tensorflow as tf
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
# Locked model: first the GitHub repo's models/ folder, otherwise Google Drive
SAVE_DIR = next((p for p in ['models/TrustBreast_locked', '../models/TrustBreast_locked']
                 if os.path.isdir(p)), None)
if SAVE_DIR is None:
    SAVE_DIR = '/content/drive/MyDrive/TrustBreast_locked'
    from google.colab import drive; drive.mount('/content/drive')
print('Loading locked model from:', SAVE_DIR)
rf_model  = joblib.load(SAVE_DIR + '/rf_model.pkl')
xgb_model = joblib.load(SAVE_DIR + '/xgb_model.pkl')
scaler    = joblib.load(SAVE_DIR + '/scaler.pkl')
dnn_best  = tf.keras.models.load_model(SAVE_DIR + '/dnn_best.keras')
dnn_model = tf.keras.models.load_model(SAVE_DIR + '/dnn_model.keras')
with open(SAVE_DIR + '/state.pkl','rb') as f: state = pickle.load(f)
globals().update({k:v for k,v in state.items() if v is not None})
if globals().get('prob_ensemble_val') is None and 'X_val_sc' in globals():
    _rf=rf_model.predict_proba(X_val_sc)[:,1]; _xg=xgb_model.predict_proba(X_val_sc)[:,1]
    _dn=dnn_best.predict(X_val_sc, verbose=0).ravel(); prob_ensemble_val=(_rf+_xg+_dn)/3
model_dnn=dnn_best; rf_aug=rf_model; xgb_aug=xgb_model; feature_names=list(X.columns)
print('LOADED locked model. Ensemble accuracy:', round(accuracy_score(y_test, ens_pred)*100,2), 'percent')

## STEP 4b — Table 2 numbers from the loaded model (99.12% check)

In [ ]:
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)
yt = np.asarray(y_test).ravel().astype(int)
def _m(prob, thr):
    pred = (np.asarray(prob).ravel() >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, pred).ravel()
    return (pred, accuracy_score(yt,pred), precision_score(yt,pred), recall_score(yt,pred),
            f1_score(yt,pred), roc_auc_score(yt,prob), tn/(tn+fp), fn)
best_rf_pred,  acc_rf,  prec_rf,  rec_rf,  f1_rf,  auc_rf,  spec_rf,  fn_rf  = _m(rf_prob_raw,  best_rf_thr)
best_xgb_pred, acc_xgb, prec_xgb, rec_xgb, f1_xgb, auc_xgb, spec_xgb, fn_xgb = _m(xgb_prob_raw, best_xgb_thr)
best_dnn_pred, acc_d,   prec_d,   rec_d,   f1_d,   auc_d,   spec_d,   fn_d   = _m(dnn_prob_raw, best_dnn_thr)
_,             acc_e,   prec_e,   rec_e,   f1_e,   auc_e,   spec_e,   fn_e   = _m(prob_ensemble, best_ens_thr)
print(f"{'Metric':<12}{'RF':>9}{'XGB':>9}{'DNN':>9}{'Ens':>9}")
for n, v in [('Acc %',[acc_rf*100,acc_xgb*100,acc_d*100,acc_e*100]),('Precision',[prec_rf,prec_xgb,prec_d,prec_e]),
             ('Recall',[rec_rf,rec_xgb,rec_d,rec_e]),('F1',[f1_rf,f1_xgb,f1_d,f1_e]),
             ('AUC',[auc_rf,auc_xgb,auc_d,auc_e]),('Specificity',[spec_rf,spec_xgb,spec_d,spec_e]),
             ('Miss (FN)',[fn_rf,fn_xgb,fn_d,fn_e])]:
    print(f"{n:<12}" + "".join(f"{x:>9.4f}" if n not in ('Acc %','Miss (FN)') else f"{x:>9.2f}" for x in v))
print(f"\nThresholds: RF {best_rf_thr:.2f} | XGB {best_xgb_thr:.2f} | DNN {best_dnn_thr:.2f} | ENS {best_ens_thr:.2f}")


## STEP 5 — McNemar + DeLong + Bootstrap CI (Section 4.1, 4.4)

In [ ]:
# ============================================================
# STATISTICAL SIGNIFICANCE — McNemar + DeLong + Bootstrap CI
# Run AFTER the LOAD cell (no retraining needed).
# Verified output:
#   McNemar  p = 0.500 (RF) / 0.250 (XGB) / 0.250 (DNN)
#   DeLong   p = 0.502 (RF) / 0.262 (XGB) / 0.306 (DNN)
#   Ensemble AUC 0.9997, DeLong 95% CI [0.9988, 1.0000]
#   Ensemble acc 99.12%, bootstrap 95% CI [97.37, 100.00]
# ============================================================
!pip install statsmodels -q
import numpy as np
from scipy.stats import norm
from sklearn.metrics import roc_auc_score, accuracy_score
from statsmodels.stats.contingency_tables import mcnemar

y_true = np.asarray(y_test).astype(int).ravel()
best_rf_pred  = (np.asarray(rf_prob_raw)  >= best_rf_thr ).astype(int)
best_xgb_pred = (np.asarray(xgb_prob_raw) >= best_xgb_thr).astype(int)
best_dnn_pred = (np.asarray(dnn_prob_raw) >= best_dnn_thr).astype(int)
ens_pred_arr  = np.asarray(ens_pred).astype(int).ravel()

def run_mcnemar(pred_other, name):
    ce = (ens_pred_arr == y_true); co = (pred_other == y_true)
    b = int(np.sum(ce & ~co)); c = int(np.sum(~ce & co))
    p = mcnemar([[0, b], [c, 0]], exact=True).pvalue
    print(f"  {name:8s} | ensemble gains {b}, loses {c}  ->  McNemar p = {p:.3f}")

def compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5*(i+j-1) + 1; i = j
    T2 = np.empty(N); T2[J] = T; return T2

def delong(prob_a, prob_b, name):
    order = np.argsort(-y_true, kind='stable')
    m = int((y_true[order] == 1).sum())
    P = np.vstack([np.asarray(prob_a)[order], np.asarray(prob_b)[order]])
    n = P.shape[1] - m
    tx = np.empty([2,m]); ty = np.empty([2,n]); tz = np.empty([2,m+n])
    for r in range(2):
        tx[r] = compute_midrank(P[r,:m]); ty[r] = compute_midrank(P[r,m:]); tz[r] = compute_midrank(P[r])
    aucs = tz[:,:m].sum(axis=1)/m/n - (m+1.0)/(2.0*n)
    v01 = (tz[:,:m]-tx)/n; v10 = 1.0-(tz[:,m:]-ty)/m
    cov = np.cov(v01)/m + np.cov(v10)/n
    var = cov[0,0]+cov[1,1]-2*cov[0,1]
    p = 1.0 if var<=0 else 2*(1-norm.cdf(abs(aucs[0]-aucs[1])/np.sqrt(var)))
    print(f"  {name:8s} | AUC ens={aucs[0]:.4f} vs {aucs[1]:.4f}  ->  DeLong p = {p:.3f}")

print("=== McNemar (ensemble vs each) ===")
for pr, nm in [(best_rf_pred,"RF"),(best_xgb_pred,"XGBoost"),(best_dnn_pred,"DNN")]:
    run_mcnemar(pr, nm)

print("\n=== DeLong (AUC comparison) ===")
for pr, nm in [(rf_prob_raw,"RF"),(xgb_prob_raw,"XGBoost"),(dnn_prob_raw,"DNN")]:
    delong(prob_ensemble, pr, nm)

order = np.argsort(-y_true, kind='stable')
m = int((y_true[order]==1).sum()); P = np.asarray(prob_ensemble)[order]
n = len(P)-m
tx = compute_midrank(P[:m]); ty = compute_midrank(P[m:]); tz = compute_midrank(P)
auc = tz[:m].sum()/m/n - (m+1.0)/(2.0*n)
v01 = (tz[:m]-tx)/n; v10 = 1.0-(tz[m:]-ty)/m
se = np.sqrt(v01.var(ddof=1)/m + v10.var(ddof=1)/n)
print(f"\n=== Ensemble AUC ===\n  AUC = {auc:.4f}   DeLong 95% CI [{max(auc-1.96*se,0):.4f}, {min(auc+1.96*se,1):.4f}]")

rs = np.random.RandomState(42); accs=[]
for _ in range(2000):
    idx = rs.randint(0, len(y_true), len(y_true))
    accs.append(accuracy_score(y_true[idx], ens_pred_arr[idx]))
print(f"\n=== Ensemble accuracy ===\n  Acc = {accuracy_score(y_true, ens_pred_arr)*100:.2f}%"
      f"   Bootstrap 95% CI [{np.percentile(accs,2.5)*100:.2f}, {np.percentile(accs,97.5)*100:.2f}]")
from sklearn.metrics import recall_score, f1_score
rs = np.random.RandomState(42); recs=[]; f1s=[]
for _ in range(2000):
    idx = rs.randint(0, len(y_true), len(y_true))
    recs.append(recall_score(y_true[idx], ens_pred_arr[idx])); f1s.append(f1_score(y_true[idx], ens_pred_arr[idx]))
print(f"  Recall bootstrap 95% CI [{np.percentile(recs,2.5):.3f}, {np.percentile(recs,97.5):.3f}]   (paper: 0.917-1.000)")
print(f"  F1     bootstrap 95% CI [{np.percentile(f1s,2.5):.3f}, {np.percentile(f1s,97.5):.3f}]   (paper: 0.957-1.000)")


## STEP 6a — 10-fold leakage-corrected CV (Section 4.2, Fig 5)

In [ ]:
# ============================================================
#  FULL-ENSEMBLE LEAKAGE-CORRECTED 10-FOLD CROSS-VALIDATION
#  10-Fold CV of the full RF + XGBoost + DNN soft-voting ensemble
#
#  Run this cell AFTER the File-1 LOAD/DATA cell
#  (X, y must already exist — the full 569-patient dataset).
#
#  Scaler + SMOTE are fit only on each fold's training portion (no leakage).
#  Each fold's DNN is seeded with keras.utils.set_random_seed(SEED + fold) -> identical on every run.
#  Full 569-patient dataset; std = sample std (ddof=1).
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.initializers import GlorotUniform
from matplotlib.patches import Patch

SEED = 42
tf.keras.utils.set_random_seed(SEED)   # FIX: also fixes the Keras 3 global RNG (python+numpy+tf+keras)

# Full dataset (569 patients)
X_arr = np.array(X)
y_arr = np.array(y)

N_FOLDS = 10
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_accs = []
oof_true, oof_prob = [], []   # out-of-fold collectors
print("=" * 60)
print(f"FULL-ENSEMBLE LEAKAGE-CORRECTED {N_FOLDS}-FOLD CV")
print("=" * 60)
print(f"Total patients used in CV: {len(y_arr)}")
print()

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_arr, y_arr), 1):
    X_tr_raw, X_va_raw = X_arr[tr_idx], X_arr[va_idx]
    y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]

    # ---- scaler fit ONLY on this fold's train portion ----
    fold_scaler = MinMaxScaler()
    X_tr_sc = fold_scaler.fit_transform(X_tr_raw)
    X_va_sc = fold_scaler.transform(X_va_raw)          # validation never seen by fit

    # ---- SMOTE ONLY on this fold's train portion ----
    sm = SMOTE(random_state=SEED)
    X_tr_sm, y_tr_sm = sm.fit_resample(X_tr_sc, y_tr)

    # ---- Random Forest ----
    rf = RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=1)
    rf.fit(X_tr_sm, y_tr_sm)
    rf_p = rf.predict_proba(X_va_sc)[:, 1]

    # ---- XGBoost ----
    xgb = XGBClassifier(learning_rate=0.01, max_depth=4, n_estimators=500,
                        subsample=0.9, colsample_bytree=0.9, random_state=SEED,
                        n_jobs=1, eval_metric='logloss', verbosity=0)
    xgb.fit(X_tr_sm, y_tr_sm)
    xgb_p = xgb.predict_proba(X_va_sc)[:, 1]

    # ---- DNN (fresh network each fold, seeded) ----
    tf.keras.utils.set_random_seed(SEED + fold)   # FIX: each fold's DNN is identical on every run
    dnn = Sequential([
        Dense(256, activation='relu', kernel_regularizer=l2(0.0005),
              kernel_initializer=GlorotUniform(seed=SEED), input_shape=(X_tr_sm.shape[1],)),
        BatchNormalization(), Dropout(0.3),
        Dense(128, activation='relu', kernel_regularizer=l2(0.0005),
              kernel_initializer=GlorotUniform(seed=SEED + 1)),
        BatchNormalization(), Dropout(0.3),
        Dense(64, activation='relu', kernel_initializer=GlorotUniform(seed=SEED + 2)),
        BatchNormalization(), Dropout(0.2),
        Dense(1, activation='sigmoid', kernel_initializer=GlorotUniform(seed=SEED + 3)),
    ])
    dnn.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy')
    dnn.fit(X_tr_sm, y_tr_sm, epochs=80, batch_size=16, verbose=0,
            validation_split=0.15,
            callbacks=[EarlyStopping(patience=12, restore_best_weights=True)])
    dnn_p = dnn.predict(X_va_sc, verbose=0).ravel()

    # ---- soft-vote ensemble (simple average, matches Objective-1) ----
    ens_p = (rf_p + xgb_p + dnn_p) / 3
    ens_pred_fold = (ens_p >= 0.5).astype(int)
    acc = accuracy_score(y_va, ens_pred_fold)
    fold_accs.append(acc)
    oof_true.extend(list(np.asarray(y_va).ravel()))
    oof_prob.extend(list(np.asarray(ens_p).ravel()))
    print(f"Fold {fold}: ensemble accuracy = {acc*100:.2f}%  "
          f"(RF {accuracy_score(y_va,(rf_p>=0.5).astype(int))*100:.2f}% | "
          f"XGB {accuracy_score(y_va,(xgb_p>=0.5).astype(int))*100:.2f}% | "
          f"DNN {accuracy_score(y_va,(dnn_p>=0.5).astype(int))*100:.2f}%)")

# ============================================================
#  RESULT  (sample std = ddof=1, as in the figure)
# ============================================================
fold_accs = np.array(fold_accs)
mean_acc = fold_accs.mean() * 100
std_acc  = fold_accs.std(ddof=1) * 100          # sample std (ddof=1)

print()
print("=" * 60)
print(f"{N_FOLDS}-FOLD CV RESULT (full ensemble, leakage-free)")
print("=" * 60)
print(f"Mean accuracy: {mean_acc:.2f}%")
print(f"Std deviation: {std_acc:.2f}%")
print(f"-> Report as: {mean_acc:.2f}% +/- {std_acc:.2f}%")

# ---- pooled out-of-fold AUC + bootstrap 95% CI (actual numbers) ----
from sklearn.metrics import roc_auc_score
oof_true = np.array(oof_true); oof_prob = np.array(oof_prob)
oof_pred = (oof_prob >= 0.5).astype(int)
pooled_auc = roc_auc_score(oof_true, oof_prob)
rng = np.random.RandomState(42); N = len(oof_true); boot = []
for _ in range(2000):
    b = rng.randint(0, N, N)
    boot.append((oof_pred[b] == oof_true[b]).mean())
ci_low, ci_high = np.percentile(boot, [2.5, 97.5]) * 100
print()
print(f"Pooled out-of-fold AUC : {pooled_auc:.4f}")
print(f"Bootstrap 95% CI       : {ci_low:.2f}% - {ci_high:.2f}%")
print()
print("=> Values for thesis Section 4.2.2:")
print(f"   {mean_acc:.2f}% +/- {std_acc:.2f}%  |  95% CI {ci_low:.2f}-{ci_high:.2f}%  |  pooled AUC {pooled_auc:.4f}")

# ============================================================
#  FIGURE 4.6  ->  bar chart (green = above mean, red = below)
# ============================================================
accs   = fold_accs * 100
folds  = [f"Fold {i+1}" for i in range(len(accs))]
colors = ['#2ca02c' if a >= mean_acc else '#e74c3c' for a in accs]

fig, ax = plt.subplots(figsize=(12, 5.5))
bars = ax.bar(folds, accs, color=colors, edgecolor='black', width=0.6, zorder=3)
ax.axhline(mean_acc, ls='--', color='#333', lw=1.5, zorder=2)

for b, a in zip(bars, accs):
    ax.text(b.get_x() + b.get_width() / 2, a + 0.12, f'{a:.2f}%',
            ha='center', va='bottom', fontweight='bold', fontsize=9)

ax.set_ylim(accs.min() - 2.0, 101.0)
ax.set_ylabel('Accuracy (%)')
ax.set_title(f"Full-Ensemble Leakage-Corrected {N_FOLDS}-Fold Cross-Validation\n"
             f"Mean = {mean_acc:.2f}% \u00b1 {std_acc:.2f}%  (RF + XGBoost + DNN soft-voting)",
             fontsize=11, fontweight='bold', pad=15)
ax.grid(axis='y', alpha=0.3, zorder=0)

ax.legend(handles=[Patch(color='#2ca02c', label='Above mean'),
                   Patch(color='#e74c3c', label='Below mean'),
                   plt.Line2D([0], [0], ls='--', color='#333',
                              label=f'Mean = {mean_acc:.2f}%')],
          loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('figure_4_6_cv_10fold.png', dpi=200, bbox_inches='tight')
plt.show()
print("\nChart saved as: figure_4_6_cv_10fold.png")

# handy variables for later cells
cv_full_ensemble_mean   = fold_accs.mean()
cv_full_ensemble_std    = fold_accs.std(ddof=1)
cv_full_ensemble_scores = fold_accs


## STEP 6b — Cross-conformal (Table 12)

In [ ]:
# ============================================================
# CROSS-CONFORMAL — all 569 patients, no reuse
# NOTE: below, first the pooled thresholds (in-sample coverage = index/n by construction),
#       then the OUT-OF-FOLD coverage used in the paper (each fold's threshold from the other 9 folds).
# Run AFTER the CV cell (requires oof_true and oof_prob)
# ============================================================
import numpy as np

y_oof = np.asarray(oof_true).ravel().astype(int)
p_oof = np.asarray(oof_prob).ravel()

# nonconformity score for each patient — from a model that never saw that patient
scores = np.where(y_oof == 1, 1 - p_oof, p_oof)

def tau_of(sc, alpha):
    sc = np.sort(sc); n = len(sc)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    return sc[min(k, n) - 1], k, n

print("=" * 62)
for alpha in (0.05, 0.10):
    t_all, k, n = tau_of(scores, alpha)
    t_b, kb, nb_ = tau_of(scores[y_oof == 0], alpha)
    t_m, km, nm  = tau_of(scores[y_oof == 1], alpha)

    inc_b = p_oof <= t_all
    inc_m = (1 - p_oof) <= t_all
    cov   = np.mean(np.where(y_oof == 1, inc_m, inc_b))
    sizes = inc_b.astype(int) + inc_m.astype(int)

    cov_b = np.mean((p_oof <= t_b)[y_oof == 0])
    cov_m = np.mean(((1 - p_oof) <= t_m)[y_oof == 1])

    print(f"alpha = {alpha:.2f}   (target coverage {100*(1-alpha):.0f}%)")
    print(f"  marginal tau = {t_all:.4f}   index {k}/{n}")
    print(f"  marginal coverage : {cov*100:.2f}%")
    print(f"  sets: {np.sum(sizes==1)} singleton | {np.sum(sizes==2)} ambiguous | {np.sum(sizes==0)} empty")
    print(f"  Mondrian benign    n={nb_:<3} index={kb:<3} tau={t_b:.4f}  "
          f"{'SATURATED' if kb>=nb_ else 'ok'}   coverage {cov_b*100:.2f}%")
    print(f"  Mondrian malignant n={nm:<3} index={km:<3} tau={t_m:.4f}  "
          f"{'SATURATED' if km>=nm else 'ok'}   coverage {cov_m*100:.2f}%")
    print("-" * 62)

# ---------- out-of-fold (fold-held-out) conformal evaluation ----------
import math, numpy as np
def kth(sc, alpha):
    sc = np.sort(sc); n = len(sc); k = math.ceil((n + 1) * (1 - alpha)); return sc[min(k, n) - 1], k, n

def conformal_report(y, p, fold, alpha, label):
    """y: labels (1 = positive class), p: out-of-fold P(positive), fold: fold id of each patient."""
    y = np.asarray(y).astype(int); p = np.asarray(p, float); fold = np.asarray(fold)
    s = np.where(y == 1, 1 - p, p)                       # nonconformity score, Eq. (4)
    # (a) in-sample: threshold and coverage from the same 569 scores (what the paper reported)
    t, k, n = kth(s, alpha); ins = np.mean(s <= t)
    # (b) out-of-fold: for each fold, threshold from the OTHER folds only, applied to this fold
    cov = np.zeros(len(y), bool); size = np.zeros(len(y), int)
    covM = np.zeros(len(y), bool)
    for f in np.unique(fold):
        te, ca = fold == f, fold != f
        tj, _, _ = kth(s[ca], alpha)                      # marginal threshold
        tb, _, _ = kth(s[ca & (y == 0)], alpha)           # Mondrian thresholds
        tm, _, _ = kth(s[ca & (y == 1)], alpha)
        inc0, inc1 = p[te] <= tj, (1 - p[te]) <= tj
        cov[te] = np.where(y[te] == 1, inc1, inc0); size[te] = inc0.astype(int) + inc1.astype(int)
        covM[te] = np.where(y[te] == 1, (1 - p[te]) <= tm, p[te] <= tb)
    print(f"{label}  alpha={alpha:.2f}")
    print(f"   in-sample (paper so far): tau {t:.4f}  index {k}/{n}  coverage {ins*100:.2f}%  (= index/n by construction)")
    print(f"   OUT-OF-FOLD marginal coverage {cov.mean()*100:.2f}%  | positive {cov[y==1].mean()*100:.2f}%  negative {cov[y==0].mean()*100:.2f}%")
    print(f"   OUT-OF-FOLD sets single/ambiguous/empty = {(size==1).sum()}/{(size==2).sum()}/{(size==0).sum()}")
    print(f"   OUT-OF-FOLD Mondrian coverage  positive {covM[y==1].mean()*100:.2f}%  negative {covM[y==0].mean()*100:.2f}%")
    return dict(ins=ins, cov=cov.mean(), cpos=cov[y==1].mean(), cneg=cov[y==0].mean(),
                sets=((size==1).sum(), (size==2).sum(), (size==0).sum()), mpos=covM[y==1].mean(), mneg=covM[y==0].mean())

fold_id = np.concatenate([np.full(len(va), f) for f, (_, va) in enumerate(skf.split(X_arr, y_arr), 1)])
CCW = {a: conformal_report(oof_true, oof_prob, fold_id, a, 'WBCD') for a in (0.05, 0.10)}
print('paper (Table 12, held-out folds): a=0.05 95.08% | sets 550/0/19 | Mondrian B 94.96, M 94.81 ; a=0.10 89.98% | 516/0/53 | Mondrian B 89.92, M 90.57')


## STEP 6c — Per-model figures (Fig 3) and model comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             RocCurveDisplay, roc_auc_score)

# ---- feature names (from X = df.drop(['id','diagnosis'])) ----
feature_names = list(X.columns)

# ---- DNN feature importance = permutation importance (AUC drop) ----
def dnn_perm_importance(model, X_sc, y_true, n_repeats=3, seed=42):
    rng = np.random.RandomState(seed)
    base = roc_auc_score(y_true, model.predict(X_sc, verbose=0).ravel())
    Xp = X_sc.copy()
    imp = np.zeros(Xp.shape[1])
    for j in range(Xp.shape[1]):
        saved = Xp[:, j].copy()
        drops = []
        for _ in range(n_repeats):
            rng.shuffle(Xp[:, j])
            a = roc_auc_score(y_true, model.predict(Xp, verbose=0).ravel())
            drops.append(base - a)
            Xp[:, j] = saved
        imp[j] = np.mean(drops)
    return imp

# ---- generic 3-panel plotter ----
def plot_model_panel(title, y_true, y_pred, y_prob, importances, fname):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

    # (1) Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Malignant']).plot(
        ax=axes[0], cmap='Greens', colorbar=False)
    axes[0].set_title(f"{title} - Confusion Matrix", fontsize=10, fontweight='bold')

    # (2) ROC curve  (single legend entry, full-precision AUC — no duplicate)
    auc = roc_auc_score(y_true, y_prob)
    try:
        disp = RocCurveDisplay.from_predictions(y_true, y_prob, ax=axes[1],
                                                curve_kwargs={'color': '#0f7d7d'})
    except (TypeError, AttributeError):   # older sklearn or older matplotlib
        disp = RocCurveDisplay.from_predictions(y_true, y_prob, ax=axes[1], color='#0f7d7d')
    disp.line_.set_label(f"{title} (AUC = {auc:.4f})")   # override auto label -> no duplicate
    axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Chance')
    axes[1].set_title(f"{title} - ROC (AUC = {auc:.4f})", fontsize=10, fontweight='bold')
    axes[1].legend(loc='lower right', fontsize=8)

    # (3) Top-10 feature importances
    order = np.argsort(importances)[::-1][:10]
    names = [feature_names[i] for i in order][::-1]
    vals  = [importances[i]   for i in order][::-1]
    axes[2].barh(names, vals, color='#0f7d7d', edgecolor='black')
    axes[2].set_title(f"{title} - Top-10 Feature Importances", fontsize=10, fontweight='bold')
    axes[2].tick_params(axis='y', labelsize=8)

    plt.tight_layout()
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")

# ===== Figure 4.1 — Random Forest =====
plot_model_panel("Random Forest", y_test, best_rf_pred, rf_prob_raw,
                 rf_model.feature_importances_, "figure_4_1_rf.png")

# ===== Figure 4.2 — XGBoost =====
plot_model_panel("XGBoost", y_test, best_xgb_pred, xgb_prob_raw,
                 xgb_model.feature_importances_, "figure_4_2_xgb.png")

# ===== Figure 4.3 — DNN (permutation importance) =====
print("Computing DNN permutation importance (this takes a while)...")
dnn_imp = dnn_perm_importance(dnn_best, X_test_sc, y_test.values if hasattr(y_test,'values') else y_test)
plot_model_panel("DNN", y_test, best_dnn_pred, dnn_prob_raw,
                 dnn_imp, "figure_4_3_dnn.png")

In [ ]:
# ============================================================
#  FIGURE 4.5 — Model Comparison (All Metrics, grouped bars)
#  Models: RF | XGBoost | DNN | Ensemble
#  Metrics: Accuracy | F1-Score | Precision | Recall
#
#  Before running: the File-1 ENSEMBLE cell must have been run so that
#  the acc_/prec_/rec_/f1_ variables exist.
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

models  = ['RF', 'XGBoost', 'DNN', 'Ensemble']
metrics = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
colors  = ['#1f9bd6', '#2ec16b', '#f28e2b', '#e74c3c']   # blue, green, orange, red

# rows = metrics, cols = models
data = np.array([
    [acc_rf,  acc_xgb,  acc_d,  acc_e ],   # Accuracy
    [f1_rf,   f1_xgb,   f1_d,   f1_e  ],   # F1-Score
    [prec_rf, prec_xgb, prec_d, prec_e],   # Precision
    [rec_rf,  rec_xgb,  rec_d,  rec_e ],   # Recall
])

x = np.arange(len(models))
n = len(metrics)
w = 0.2

fig, ax = plt.subplots(figsize=(10, 5.5))
for i, (m, col) in enumerate(zip(metrics, colors)):
    ax.bar(x + (i - (n - 1) / 2) * w, data[i], width=w, label=m,
           color=col, edgecolor='black', linewidth=0.4, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0.86, 1.02)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — All Metrics', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.legend(loc='upper right', fontsize=8, ncol=1)

plt.tight_layout()
plt.savefig('figure_4_5_model_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: figure_4_5_model_comparison.png")

# quick text summary
print("\nMetric values used:")
print(f"{'Model':<10}{'Acc':>8}{'F1':>8}{'Prec':>8}{'Recall':>8}{'AUC':>8}")
for j, mdl in enumerate(models):
    aucs = [auc_rf, auc_xgb, auc_d, auc_e]
    print(f"{mdl:<10}{data[0][j]:>8.4f}{data[1][j]:>8.4f}{data[2][j]:>8.4f}{data[3][j]:>8.4f}{aucs[j]:>8.4f}")


## STEP 7 — ★ FIXED leakage test (Table 3) — ~1 hour
In the previous version, synthetic SMOTE rows were also included in the test folds. Evaluation is now on real patients only (as in File 5 FIXED).

In [ ]:
# ============================================================================
# FIX A — TABLE 3: leakage on WBCD, full 3-model ensemble, REAL-PATIENT evaluation
# File 1 — run after the LOAD/DATA + CV cells (requires X_arr, y_arr). Replaces the old cell 22.
# ============================================================================
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from scipy import stats

N_FOLDS, SEED, N_REPEATS = 10, 42, 5

def build_dnn(input_dim, seed):
    tf.keras.utils.set_random_seed(seed)
    m = tf.keras.Sequential([tf.keras.layers.Input(shape=(input_dim,))])
    for u, dr in [(1024,.4),(512,.4),(256,.3),(128,.3),(64,.2),(32,.2)]:
        m.add(tf.keras.layers.Dense(u, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(5e-4)))
        m.add(tf.keras.layers.BatchNormalization()); m.add(tf.keras.layers.Dropout(dr))
    m.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    m.compile(optimizer='adam', loss='binary_crossentropy'); return m

def ensemble_prob(Xtr, ytr, Xva, seed):
    rf  = RandomForestClassifier(n_estimators=500, random_state=seed, n_jobs=-1).fit(Xtr, ytr)
    xgb = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.01, subsample=0.9,
                        colsample_bytree=0.9, random_state=seed, eval_metric='logloss',
                        verbosity=0).fit(Xtr, ytr)
    dnn = build_dnn(Xtr.shape[1], seed); dnn.fit(Xtr, ytr, epochs=60, batch_size=32, verbose=0)
    return (rf.predict_proba(Xva)[:,1] + xgb.predict_proba(Xva)[:,1]
            + dnn.predict(Xva, verbose=0).ravel()) / 3

def run_protocol(X, y, leak, seed):
    n_real = len(y)
    if leak:
        Xs = MinMaxScaler().fit_transform(X)                  # scaler on the full data (incorrect practice)
        Xr, yr = SMOTE(random_state=seed).fit_resample(Xs, y) # SMOTE BEFORE fold creation
        assert np.allclose(Xr[:n_real], Xs)                   # imblearn: real rows first, then synthetic
    else:
        Xr, yr = X, y
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    accs, ot, op = [], [], []
    for tr, va in skf.split(Xr, yr):
        if leak:
            va = va[va < n_real]                              # ★ FIX: test on REAL patients only
            Xtr, ytr, Xva, yva = Xr[tr], yr[tr], Xr[va], yr[va]
        else:
            sc = MinMaxScaler().fit(Xr[tr])
            Xtr, Xva = sc.transform(Xr[tr]), sc.transform(Xr[va])
            Xtr, ytr = SMOTE(random_state=seed).fit_resample(Xtr, yr[tr]); yva = yr[va]
        p = ensemble_prob(Xtr, ytr, Xva, seed)
        accs.append(accuracy_score(yva, (p >= .5).astype(int))); ot.extend(yva); op.extend(p)
    return np.array(accs), roc_auc_score(ot, op)

X_ = np.asarray(X_arr, float); y_ = np.asarray(y_arr).ravel().astype(int)
L, C, Lu, Cu = [], [], [], []
for r in range(N_REPEATS):
    a, u = run_protocol(X_, y_, True,  SEED + r); L += list(a); Lu.append(u)
    a, u = run_protocol(X_, y_, False, SEED + r); C += list(a); Cu.append(u)
    print(f"  repeat {r+1}/{N_REPEATS} done")

L, C = np.array(L)*100, np.array(C)*100
d = L - C
v = d.var(ddof=1) * (1/len(d) + (1/N_FOLDS)/(1-1/N_FOLDS))       # Nadeau-Bengio
t = d.mean()/np.sqrt(v); p = 2*(1 - stats.t.cdf(abs(t), len(d)-1))
half = stats.t.ppf(.975, len(d)-1)*np.sqrt(v)

print("\n" + "="*74)
print(f"{'Protocol':<46}{'CV accuracy (%)':>16}{'Pooled AUC':>12}")
print("-"*74)
print(f"{'SMOTE before fold creation (common practice)':<46}{L.mean():>9.2f} ± {L.std(ddof=1):<4.2f}{np.mean(Lu):>12.4f}")
print(f"{'SMOTE inside each training fold (ours)':<46}{C.mean():>9.2f} ± {C.std(ddof=1):<4.2f}{np.mean(Cu):>12.4f}")
print("-"*74)
print(f"Difference: {d.mean():+.2f} points, 95% CI [{d.mean()-half:+.2f}, {d.mean()+half:+.2f}]")
print(f"Nadeau-Bengio t = {t:.2f}, p = {p:.3f}   ({len(d)} folds)")
print(f"Errors removed by leakage: {d.mean()/(100-C.mean())*100:.1f}% of the honest pipeline's errors")
print("="*74)
pd.DataFrame({'leaky':L,'correct':C}).to_csv('table3_leakage_WBCD_fixed.csv', index=False)

T3 = dict(L=L, C=C, Lu=Lu, Cu=Cu, d=d, t=t, p=p, half=half)   # stored for the summary


## STEP 8 — ★ FIXED calibration baselines (Table 10) — ~20–40 min
The previous ECE formula had a bug (it gave ECE ≈ 0.63). ECE is now computed correctly (confidence vs. accuracy).

In [ ]:
# ============================================================================
# STEP 7 — MODERN UQ BASELINES
#   deep ensemble (5 nets) · temperature scaling · Brier · equal-mass ECE + CI
# Run after the LOAD cell. Runtime: ~20-40 minutes.
# Variable names are auto-detected from File 1.
# ============================================================================
import numpy as np, tensorflow as tf
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
from scipy.optimize import minimize_scalar

G = globals()
def pick(*names):
    for n in names:
        if n in G and G[n] is not None: return G[n], n
    return None, None

Xtr, ntr = pick('X_train_aug','X_train_sm','X_fit_sc','X_train_sc')
ytr, _   = pick('y_train_aug','y_train_sm','y_fit','y_train')
Xte, nte = pick('X_test_sc','X_test_arr')
yte_r, _ = pick('y_test')
Xva, _   = pick('X_val_sc')
yva_r, _ = pick('y_val')
pv, npv  = pick('prob_ensemble_val')

y_te = np.asarray(yte_r).ravel().astype(int)
p_ens = np.asarray(prob_ensemble).ravel()
print(f"train: {ntr} {np.shape(Xtr)} | test: {nte} {np.shape(Xte)} | val prob: {npv}")

# ---------------- metrics (FIXED) ----------------
def _conf_correct(y, p):
    y = np.asarray(y).astype(int).ravel(); p = np.asarray(p, float).ravel()
    pred = (p >= .5).astype(int)
    conf = np.where(pred == 1, p, 1 - p)          # confidence of the predicted class
    return conf, (pred == y).astype(float)

def ece_width(y, p, n_bins=10):                   # identical to File 3 compute_ece
    conf, corr = _conf_correct(y, p); e = m = 0.0
    edges = np.linspace(0, 1, n_bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        k = (conf > lo) & (conf <= hi)
        if k.any():
            g = abs(corr[k].mean() - conf[k].mean()); e += k.mean()*g; m = max(m, g)
    return e, m

def ece_mass(y, p, n_bins=10):
    conf, corr = _conf_correct(y, p); e = 0.0
    for b in np.array_split(np.argsort(conf, kind='stable'), n_bins):
        if len(b): e += len(b)/len(conf) * abs(corr[b].mean() - conf[b].mean())
    return e

def boot_ci(fn, y, p, n=2000, seed=42):
    y = np.asarray(y); p = np.asarray(p); rs = np.random.RandomState(seed)
    v = [fn(y[i], p[i]) for i in (rs.randint(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(v, [2.5, 97.5])

# ---------------- deep ensemble ----------------
def build(dim, seed):
    tf.keras.utils.set_random_seed(seed)
    m = tf.keras.Sequential([tf.keras.layers.Input(shape=(dim,))])
    for u, dr in [(1024,.4),(512,.4),(256,.3),(128,.3),(64,.2),(32,.2)]:
        m.add(tf.keras.layers.Dense(u, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(5e-4)))
        m.add(tf.keras.layers.BatchNormalization()); m.add(tf.keras.layers.Dropout(dr))
    m.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    m.compile(optimizer='adam', loss='binary_crossentropy'); return m

print("\nTraining deep ensemble (5 networks)...")
mem = []
for k in range(5):
    d = build(np.shape(Xtr)[1], 100+k)
    d.fit(np.asarray(Xtr), np.asarray(ytr).ravel(), epochs=60, batch_size=32, verbose=0)
    mem.append(d.predict(np.asarray(Xte), verbose=0).ravel())
    print(f"  member {k+1}/5")
de_mean, de_std = np.mean(mem, axis=0), np.std(mem, axis=0)

# ---------------- temperature scaling ----------------
def logit(p):
    p = np.clip(np.asarray(p, float), 1e-7, 1-1e-7); return np.log(p/(1-p))

p_temp, T = None, None
if pv is not None and yva_r is not None:
    yv = np.asarray(yva_r).ravel().astype(int); lv = logit(np.asarray(pv).ravel())
    def nll(t):
        q = np.clip(1/(1+np.exp(-lv/t)), 1e-7, 1-1e-7)
        return -np.mean(yv*np.log(q) + (1-yv)*np.log(1-q))
    T = minimize_scalar(nll, bounds=(0.05,10), method='bounded').x
    p_temp = 1/(1+np.exp(-logit(p_ens)/T))
    print(f"\nTemperature fitted on {len(yv)} validation patients: T = {T:.3f}")
else:
    print("\n(validation probabilities not found — skipping temperature scaling)")

# ---------------- comparison (FIXED) ----------------
import pandas as pd
rows = [('Soft-voting ensemble', p_ens), ('Deep ensemble (5 networks)', de_mean)]
if p_temp is not None: rows.append((f'Ensemble + temperature (T = {T:.3f})', p_temp))

out = []
print(f"{'Method':<34}{'Acc':>7}{'AUC':>8}{'Brier':>8}{'ECE-w':>8}{'ECE-m':>8}{'MCE':>8}{'ECE-m 95% CI':>20}")
print("-"*101)
for nm, p in rows:
    p = np.asarray(p, float).ravel()
    ew, mce = ece_width(y_te, p); em = ece_mass(y_te, p)
    lo, hi = boot_ci(ece_mass, y_te, p)
    acc = accuracy_score(y_te, (p >= .5).astype(int))*100
    auc = roc_auc_score(y_te, p); br = brier_score_loss(y_te, p)
    out.append([nm, acc, auc, br, ew, em, mce, lo, hi])
    print(f"{nm:<34}{acc:>7.2f}{auc:>8.4f}{br:>8.4f}{ew:>8.4f}{em:>8.4f}{mce:>8.4f}   [{lo:.4f}, {hi:.4f}]")
pd.DataFrame(out, columns=['method','acc','auc','brier','ece_width','ece_mass','mce','ci_lo','ci_hi'])\
  .to_csv('table10_calibration_fixed.csv', index=False)
print("\nRewrite the Paper Table 10 + Section 4.8.1 numbers (47%, 'fivefold', etc.) from THIS output.")

T10 = dict(out=out, T=T)   # stored for the summary


## STEP 9 — 30 repeated splits (Table 5) — ~1–2 hours

In [ ]:
# ============================================================================
# STEP 2 — REPEATED SPLITS
# 99.12% comes from a single split (random_state=42). This runs 30 splits
# to give a proper mean ± SD. Variable names are auto-detected.
#
# Runtime: ~1-2 hours.  To shorten, set N_SPLITS = 10.
# ============================================================================
N_SPLITS = 30

import numpy as np, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

G = globals()
def pick(*names):
    for n in names:
        if n in G and G[n] is not None: return np.asarray(G[n])
    raise NameError(f"none of {names} found")

X_all = pick('X_arr').astype(float)
y_all = pick('y_arr').ravel().astype(int)
print(f"data: {X_all.shape}, malignant {y_all.sum()}/{len(y_all)}\n")

def build(dim, seed):
    tf.keras.utils.set_random_seed(seed)
    m = tf.keras.Sequential([tf.keras.layers.Input(shape=(dim,))])
    for u, dr in [(1024,.4),(512,.4),(256,.3),(128,.3),(64,.2),(32,.2)]:
        m.add(tf.keras.layers.Dense(u, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(5e-4)))
        m.add(tf.keras.layers.BatchNormalization()); m.add(tf.keras.layers.Dropout(dr))
    m.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    m.compile(optimizer='adam', loss='binary_crossentropy'); return m

acc, auc, sens, fps, fns = [], [], [], [], []
for k in range(N_SPLITS):
    Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.2,
                                          stratify=y_all, random_state=k)
    sc = MinMaxScaler().fit(Xtr)
    Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)
    Xr, yr = SMOTE(random_state=k).fit_resample(Xtr_s, ytr)

    rf = RandomForestClassifier(n_estimators=500, random_state=k, n_jobs=-1).fit(Xr, yr)
    xg = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.01, subsample=0.9,
                       colsample_bytree=0.9, random_state=k, eval_metric='logloss',
                       verbosity=0).fit(Xr, yr)
    dn = build(Xr.shape[1], k); dn.fit(Xr, yr, epochs=60, batch_size=32, verbose=0)

    p = (rf.predict_proba(Xte_s)[:,1] + xg.predict_proba(Xte_s)[:,1]
         + dn.predict(Xte_s, verbose=0).ravel()) / 3
    yp = (p >= .5).astype(int)
    acc.append(accuracy_score(yte, yp)); auc.append(roc_auc_score(yte, p))
    sens.append(recall_score(yte, yp))
    fps.append(int(((yp==1)&(yte==0)).sum())); fns.append(int(((yp==0)&(yte==1)).sum()))
    if (k+1) % 5 == 0: print(f"  split {k+1}/{N_SPLITS}  (running mean "
                             f"{np.mean(acc)*100:.2f}%)")

acc, auc, sens = np.array(acc)*100, np.array(auc), np.array(sens)*100
pct = (acc < 99.12).mean()*100

print("\n" + "="*72)
print(f"  {N_SPLITS} REPEATED STRATIFIED 80/20 SPLITS")
print("="*72)
print(f"  Accuracy    : {acc.mean():.2f} ± {acc.std(ddof=1):.2f}    "
      f"range [{acc.min():.2f}, {acc.max():.2f}]")
print(f"  AUC-ROC     : {auc.mean():.4f} ± {auc.std(ddof=1):.4f}")
print(f"  Sensitivity : {sens.mean():.2f} ± {sens.std(ddof=1):.2f}")
print(f"  Median      : {np.median(acc):.2f}    IQR "
      f"[{np.percentile(acc,25):.2f}, {np.percentile(acc,75):.2f}]")
print("-"*72)
print(f"  False positives : mean {np.mean(fps):.2f}/split, max {max(fps)}, "
      f"zero-FP splits {sum(1 for f in fps if f==0)}/{N_SPLITS}")
print(f"  False negatives : mean {np.mean(fns):.2f}/split, max {max(fns)}")
print("-"*72)
print(f"  Single split (random_state=42) = 99.12%")
print(f"  Repeated mean                  = {acc.mean():.2f}%   "
      f"(difference {99.12-acc.mean():+.2f} points)")
print(f"  99.12% lies at the {pct:.0f}th percentile of this distribution")
print(f"  Splits that gave 99.12% or better: "
      f"{int((acc>=99.12).sum())}/{N_SPLITS}")
print("="*72)

import pandas as pd
pd.DataFrame({'split_seed':range(len(acc)),'accuracy':acc,'auc':auc,'sensitivity':sens,
              'false_pos':fps,'false_neg':fns}).to_csv('table5_repeated_splits.csv', index=False)
rep_acc, rep_auc, rep_sens, rep_fps, rep_fns = acc, auc, sens, fps, fns


## STEP 10 — Baselines, same protocol (Table 6) — ~15–25 min

In [ ]:
# ============================================================================
# STEP 4 — BASELINES: "is this much complexity justified?"
# In File 1, run AFTER the CV cell.   Runtime: ~15-25 minutes
#
# Everything uses EXACTLY THE SAME leakage-free protocol (same folds, same seeds).
# ============================================================================
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from scipy import stats

N_FOLDS, SEED, N_REPEATS = 10, 42, 3
X_all = np.asarray(X_arr, dtype=float)
y_all = np.asarray(y_arr).ravel().astype(int)

def dnn(dim, seed):
    import tensorflow as tf
    tf.keras.utils.set_random_seed(seed)
    m = tf.keras.Sequential([tf.keras.layers.Input(shape=(dim,))])
    for u, dr in [(1024,.4),(512,.4),(256,.3),(128,.3),(64,.2),(32,.2)]:
        m.add(tf.keras.layers.Dense(u, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(5e-4)))
        m.add(tf.keras.layers.BatchNormalization()); m.add(tf.keras.layers.Dropout(dr))
    m.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    m.compile(optimizer='adam', loss='binary_crossentropy'); return m

def fit_predict(name, Xtr, ytr, Xva, seed):
    if name == 'Logistic regression (L2)':
        return LogisticRegression(penalty='l2', C=1.0, max_iter=5000,
                                  random_state=seed).fit(Xtr,ytr).predict_proba(Xva)[:,1]
    if name == 'SVM + LDA (Omondiagbe-style)':
        p = make_pipeline(LinearDiscriminantAnalysis(n_components=1),
                          SVC(probability=True, random_state=seed)).fit(Xtr,ytr)
        return p.predict_proba(Xva)[:,1]
    if name == 'Random forest':
        return RandomForestClassifier(n_estimators=500, random_state=seed,
                                      n_jobs=-1).fit(Xtr,ytr).predict_proba(Xva)[:,1]
    if name == 'XGBoost (tuned)':
        return XGBClassifier(n_estimators=800, max_depth=3, learning_rate=0.03,
                             subsample=0.9, colsample_bytree=0.8, reg_lambda=2.0,
                             random_state=seed, eval_metric='logloss',
                             verbosity=0).fit(Xtr,ytr).predict_proba(Xva)[:,1]
    if name == 'DNN alone':
        d = dnn(Xtr.shape[1], seed); d.fit(Xtr,ytr,epochs=60,batch_size=32,verbose=0)
        return d.predict(Xva, verbose=0).ravel()
    if name == 'Ensemble (ours)':
        rf = RandomForestClassifier(n_estimators=500, random_state=seed,
                                    n_jobs=-1).fit(Xtr,ytr).predict_proba(Xva)[:,1]
        xg = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.01, subsample=0.9,
                           colsample_bytree=0.9, random_state=seed, eval_metric='logloss',
                           verbosity=0).fit(Xtr,ytr).predict_proba(Xva)[:,1]
        d = dnn(Xtr.shape[1], seed); d.fit(Xtr,ytr,epochs=60,batch_size=32,verbose=0)
        return (rf + xg + d.predict(Xva, verbose=0).ravel()) / 3

MODELS = ['Logistic regression (L2)','SVM + LDA (Omondiagbe-style)','Random forest',
          'XGBoost (tuned)','DNN alone','Ensemble (ours)']
res = {m: {'acc': [], 'ot': [], 'op': []} for m in MODELS}

for r in range(N_REPEATS):
    sd = SEED + r
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=sd)
    for tr, va in skf.split(X_all, y_all):
        sc = MinMaxScaler().fit(X_all[tr])
        Xtr, Xva = sc.transform(X_all[tr]), sc.transform(X_all[va])
        Xtr2, ytr2 = SMOTE(random_state=sd).fit_resample(Xtr, y_all[tr])
        for m in MODELS:
            p = fit_predict(m, Xtr2, ytr2, Xva, sd)
            res[m]['acc'].append(accuracy_score(y_all[va], (p >= .5).astype(int)))
            res[m]['ot'].extend(y_all[va]); res[m]['op'].extend(p)
    print(f"  repeat {r+1}/{N_REPEATS} done")

print("\n" + "="*76)
print(f"{'Model':<32}{'CV accuracy (%)':>20}{'Pooled AUC':>12}{'vs ours':>12}")
print("-"*76)
ens = np.array(res['Ensemble (ours)']['acc'])*100
for m in MODELS:
    a = np.array(res[m]['acc'])*100
    u = roc_auc_score(res[m]['ot'], res[m]['op'])
    if m == 'Ensemble (ours)':
        d = "—"
    else:
        diff = ens.mean() - a.mean()
        dd = ens - a
        nt, ntr = 1/N_FOLDS, 1-1/N_FOLDS
        v = dd.var(ddof=1)*(1/len(dd) + nt/ntr)
        pv = 2*(1-stats.t.cdf(abs(dd.mean()/np.sqrt(v)), df=len(dd)-1)) if v>0 else 1.0
        d = f"{diff:+.2f} (p={pv:.3f})"
    print(f"{m:<32}{a.mean():>13.2f} ± {a.std(ddof=1):<4.2f}{u:>12.4f}{d:>12}")
print("="*76)
print("If logistic regression reaches ~98%, this must be stated clearly —")
print("the ensemble's advantage is only one or two patients, and that should be reported honestly.")

import pandas as pd
base_table = pd.DataFrame([{'model':m, 'cv_acc_mean':np.mean(res[m]['acc'])*100,
               'cv_acc_sd':np.std(res[m]['acc'], ddof=1)*100,
               'pooled_auc':roc_auc_score(res[m]['ot'], res[m]['op'])} for m in MODELS])
base_table.to_csv('table6_baselines.csv', index=False)


## STEP 11 — ★ RESULTS SUMMARY
**Copy and share the full output of this cell only.** It prints all of the results above in one place.

In [ ]:
import numpy as np
from scipy import stats
def show(title, fn):
    print("\n" + "="*70 + f"\n{title}\n" + "="*70)
    try: fn()
    except Exception as e: print(f"  (not found — was this step run? error: {str(e)[:80]})")

show("T2  Held-out test set (paper Table 2)", lambda: print(
    f"  Ensemble acc {acc_e*100:.2f}% | AUC {auc_e:.4f} | recall {rec_e:.4f} | spec {spec_e:.4f} | FN {fn_e}\n"
    f"  RF {acc_rf*100:.2f} | XGB {acc_xgb*100:.2f} | DNN {acc_d*100:.2f}"))
show("CV  10-fold full ensemble (Section 4.2)", lambda: print(
    f"  {cv_full_ensemble_mean*100:.2f} ± {cv_full_ensemble_std*100:.2f} | pooled AUC {pooled_auc:.4f} | CI {ci_low:.2f}-{ci_high:.2f}"))
def _t3():
    L, C, Lu, Cu, d, t, p, half = (T3[k] for k in ('L','C','Lu','Cu','d','t','p','half'))
    print(f"  leaky {L.mean():.2f} ± {L.std(ddof=1):.2f} (AUC {np.mean(Lu):.4f})")
    print(f"  correct {C.mean():.2f} ± {C.std(ddof=1):.2f} (AUC {np.mean(Cu):.4f})")
    print(f"  Δ {d.mean():+.2f} [{d.mean()-half:+.2f}, {d.mean()+half:+.2f}]  t={t:.2f} p={p:.3f}")
    print(f"  errors removed {d.mean()/(100-C.mean())*100:.1f}%")
show("T3  Leakage WBCD — FIXED (Table 3)", _t3)
def _t10():
    for r in T10['out']: print(f"  {r[0]:<34} acc {r[1]:.2f} AUC {r[2]:.4f} Brier {r[3]:.4f} "
                        f"ECE-w {r[4]:.4f} ECE-m {r[5]:.4f} MCE {r[6]:.4f} CI [{r[7]:.4f},{r[8]:.4f}]")
    print(f"  Temperature T = {T10['T']:.3f}")
show("T10 Calibration — FIXED ECE (Table 10)", _t10)
def _t5():
    a = np.asarray(rep_acc)
    print(f"  acc {a.mean():.2f} ± {a.std(ddof=1):.2f} range [{a.min():.2f}, {a.max():.2f}]")
    print(f"  median {np.median(a):.2f} IQR [{np.percentile(a,25):.2f}, {np.percentile(a,75):.2f}]")
    print(f"  AUC {np.mean(rep_auc):.4f} ± {np.std(rep_auc,ddof=1):.4f} | sens {np.mean(rep_sens):.2f} ± {np.std(rep_sens,ddof=1):.2f}")
    print(f"  FP mean {np.mean(rep_fps):.2f} max {max(rep_fps)} zero-FP {sum(f==0 for f in rep_fps)}/{len(a)} | FN mean {np.mean(rep_fns):.2f} max {max(rep_fns)}")
    print(f"  99.12% percentile {(a < 99.12).mean()*100:.0f}th | splits >= 99.12: {(a >= 99.12).sum()}/{len(a)}")
show("T5  30 repeated splits (Table 5)", _t5)
show("T6  Baselines (Table 6)", lambda: print(base_table.round(4).to_string(index=False)))


In [ ]:
# ★ REPRODUCIBILITY CHECK — comparison with the paper's numbers
import numpy as np
def r2(x): return round(float(x), 2)
def r4(x): return round(float(x), 4)
now, exp = {}, {}
def add(k, fn, e):
    try: now[k] = fn()
    except Exception as ex: now[k] = f"(not found: {str(ex)[:40]})"
    exp[k] = e
add("T2 ensemble acc / AUC",         lambda: (r2(acc_e*100), r4(auc_e)), (99.12, 0.9997))
add("CV mean / sd / pooled AUC",     lambda: (r2(cv_full_ensemble_mean*100), r2(cv_full_ensemble_std*100), r4(pooled_auc)), (96.49, 2.74, 0.9945))
add("T3 leaky / correct / Δ / p",    lambda: (r2(T3['L'].mean()), r2(T3['C'].mean()), r2(T3['d'].mean()), round(float(T3['p']),3)), (97.79, 97.33, 0.46, 0.667))
add("T10 ECE-w ens / DE / temp",     lambda: tuple(r4(r[4]) for r in T10['out']), (0.0384, 0.0330, 0.0205))
add("T10 ECE-m ens / DE / temp",     lambda: tuple(r4(r[5]) for r in T10['out']), (0.0384, 0.0065, 0.0205))
add("T5 repeated mean / sd",         lambda: (r2(np.mean(rep_acc)), r2(np.std(rep_acc, ddof=1))), (97.19, 1.65))
add("T6 LR / ensemble CV acc",       lambda: (r2(base_table.set_index('model').loc['Logistic regression (L2)','cv_acc_mean']),
                                              r2(base_table.set_index('model').loc['Ensemble (ours)','cv_acc_mean'])), (97.48, 97.31))
print("="*72 + "\nREPRODUCIBILITY CHECK (paper numbers)\n" + "="*72)
ok_all = True
for k in exp:
    ok = now[k] == exp[k]; ok_all &= ok
    print(f"  {'✅' if ok else '❌'} {k:<30} now {now[k]}   paper {exp[k]}")
print("\nALL MATCH ✅ — notebook 01 reproduces the paper" if ok_all else "\nSome numbers differ ❌")


## (OPTIONAL) Drop-column test — Table 8
Its numbers already match the paper, so **running it is not required** (~20 min). To run it, execute the cell below.

In [ ]:
# ============================================================================
# STEP 5 — is the texture_worst finding real or a SHAP artifact?
#
# In File 1, paste into a new cell AFTER the CV cell.
# Three independent tests, all addressing the same question.
# Runtime: about 15-25 minutes.
# ============================================================================
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

N_FOLDS, SEED, N_BOOT = 10, 42, 2000
FEATS = list(feature_names)
X_all = np.asarray(X_arr, dtype=float)
y_all = np.asarray(y_arr).ravel().astype(int)

# ---------- 0. which features is texture_worst correlated with? ----------
print("=" * 72)
print("  0. CORRELATION — texture_worst vs. other features")
print("=" * 72)
ti = FEATS.index('texture_worst')
cors = sorted(((abs(np.corrcoef(X_all[:, ti], X_all[:, j])[0, 1]), FEATS[j])
               for j in range(len(FEATS)) if j != ti), reverse=True)[:5]
for r, nm in cors:
    print(f"   |r| = {r:.3f}   {nm}")
print("   (if any r > 0.9, the objection that SHAP credit is split is justified)")

# ---------- helper: leakage-free CV with a given feature set ----------
def cv_score(cols, seed=SEED, with_dnn=True):
    Xc = X_all[:, cols]
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    accs, ot, op = [], [], []
    for tr, va in skf.split(Xc, y_all):
        sc = MinMaxScaler().fit(Xc[tr])
        Xtr, Xva = sc.transform(Xc[tr]), sc.transform(Xc[va])
        Xtr, ytr = SMOTE(random_state=seed).fit_resample(Xtr, y_all[tr])
        rf = RandomForestClassifier(n_estimators=500, random_state=seed, n_jobs=-1).fit(Xtr, ytr)
        xg = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.01, subsample=0.9,
                           colsample_bytree=0.9, random_state=seed, eval_metric='logloss',
                           verbosity=0).fit(Xtr, ytr)
        p = (rf.predict_proba(Xva)[:, 1] + xg.predict_proba(Xva)[:, 1]) / 2
        if with_dnn:
            import tensorflow as tf
            tf.keras.utils.set_random_seed(seed)
            d = tf.keras.Sequential([tf.keras.layers.Input(shape=(Xtr.shape[1],))])
            for u, dr in [(256, .3), (128, .3), (64, .2)]:
                d.add(tf.keras.layers.Dense(u, activation='relu'))
                d.add(tf.keras.layers.BatchNormalization())
                d.add(tf.keras.layers.Dropout(dr))
            d.add(tf.keras.layers.Dense(1, activation='sigmoid'))
            d.compile(optimizer='adam', loss='binary_crossentropy')
            d.fit(Xtr, ytr, epochs=50, batch_size=32, verbose=0)
            p = (p * 2 + d.predict(Xva, verbose=0).ravel()) / 3
        accs.append(accuracy_score(y_all[va], (p >= 0.5).astype(int)))
        ot.extend(y_all[va]); op.extend(p)
    return np.array(accs), roc_auc_score(ot, op), np.array(ot), np.array(op)

allc = list(range(len(FEATS)))

# ---------- 1. DROP-COLUMN TEST ----------
print("\n" + "=" * 72)
print("  1. DROP-COLUMN — retrain without texture_worst")
print("=" * 72)
base_acc, base_auc, ot0, op0 = cv_score(allc)
drop_acc, drop_auc, ot1, op1 = cv_score([c for c in allc if c != ti])

rng = np.random.RandomState(SEED)
diffs = []
for _ in range(N_BOOT):
    b = rng.randint(0, len(ot0), len(ot0))
    diffs.append(roc_auc_score(ot0[b], op0[b]) - roc_auc_score(ot1[b], op1[b]))
lo, hi = np.percentile(diffs, [2.5, 97.5])

print(f"   full model      : acc {base_acc.mean()*100:.2f}% ± {base_acc.std(ddof=1)*100:.2f}   AUC {base_auc:.4f}")
print(f"   texture_worst   : acc {drop_acc.mean()*100:.2f}% ± {drop_acc.std(ddof=1)*100:.2f}   AUC {drop_auc:.4f}")
print(f"   removal effect  : AUC {base_auc-drop_auc:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]")
print(f"   -> {'FINDING ROBUST (CI above zero)' if lo > 0 else 'FINDING WEAK (CI includes zero)'}")

# ---------- 2. for comparison: also remove the other top features ----------
print("\n" + "=" * 72)
print("  2. COMPARISON — how much does removing each top-5 feature change?")
print("=" * 72)
for nm in ['perimeter_worst', 'area_worst', 'concave_points_mean', 'texture_worst', 'radius_worst']:
    j = FEATS.index(nm)
    a, u, _, _ = cv_score([c for c in allc if c != j])
    print(f"   {nm:<22} AUC {u:.4f}   effect {base_auc-u:+.4f}   acc {a.mean()*100:.2f}%")

# ---------- 3. PERMUTATION IMPORTANCE (model-agnostic cross-check) ----------
print("\n" + "=" * 72)
print("  3. PERMUTATION IMPORTANCE — a perspective independent of SHAP")
print("=" * 72)
sc = MinMaxScaler().fit(X_all)
Xs = sc.transform(X_all)
Xr, yr = SMOTE(random_state=SEED).fit_resample(Xs, y_all)
rf = RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1).fit(Xr, yr)
pi = permutation_importance(rf, Xs, y_all, n_repeats=30, random_state=SEED, scoring='roc_auc')
ordr = np.argsort(-pi.importances_mean)
for r, j in enumerate(ordr[:10], 1):
    mark = "  <-- texture_worst" if j == ti else ""
    print(f"   {r:>2}. {FEATS[j]:<24} {pi.importances_mean[j]:.5f} ± {pi.importances_std[j]:.5f}{mark}")
print(f"\n   texture_worst permutation rank: {list(ordr).index(ti)+1} / {len(FEATS)}")
print("   (SHAP rank was 4 — if it also ranks high here, the cross-method evidence is strong)")


# ---- FAMILY-LEVEL DROP TEST (paper Table 8) ----
groups = {
 'None (full model)'   : [],
 'worst texture'       : ['texture_worst'],
 'mean texture'        : ['texture_mean'],
 'All texture features': ['texture_worst', 'texture_mean', 'texture_se'],
 'All size features'   : ['radius_worst', 'perimeter_worst', 'area_worst',
                          'radius_mean', 'perimeter_mean', 'area_mean'],
}
print("\n" + "=" * 72)
print(f"  {'Features removed':<24}{'Count':>7}{'Pooled AUC':>13}{'CV accuracy (%)':>18}")
print("-" * 72)
for name, drop in groups.items():
    idx = [i for i, f in enumerate(FEATS) if f not in drop]
    a, u, _, _ = cv_score(idx)
    print(f"  {name:<24}{len(drop):>7}{u:>13.4f}{a.mean()*100:>18.2f}")
print("=" * 72)
